In [ ]:
import sys
sys.path.append('../../../')
from pathlib import Path
import h5py
import numpy as np
import jax.numpy as jnp
import sys
import os
import torch
from lucid.geometry import generate_detector
from lucid.production.data_prod_utils import read_multi_event_file, print_event_info, get_particle_name, get_track_hits
from lucid.visualization import create_detector_display
import matplotlib.pyplot as plt

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

figures_dir = Path('figures')
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
filename = '/sdf/data/neutrino/cjesus/photonsim_output/water/uniform_energy/multiparticle/config_000004/events_job_000001.h5'
all_events = read_multi_event_file(filename, verbose=False)
indices, Q, T = get_track_hits(all_events[0], 1)
detector_display = create_detector_display('../../../config/SK_geom_config.json')
detector_display(indices, Q, T, file_name=None, plot_time=False, perc_min=0.0, log_scale=False)
detector_display(indices, Q, T, file_name=None, plot_time=True, perc_min=0.0, perc_max=50.0)

In [ ]:
import matplotlib.colors as mcolors
import numpy as np
from matplotlib.colors import ListedColormap, BoundaryNorm


def prepare_event_tracks(event):
    n_tracks = event["Q"].shape[0]

    track_indices = []
    for t in range(n_tracks):
        idx, Qf, Tf = get_track_hits(event, t)
        track_indices.append(idx)

    # Find overlaps
    all_idx = np.concatenate(track_indices)
    unique, counts = np.unique(all_idx, return_counts=True)
    overlap = unique[counts > 1]

    return track_indices, overlap


def build_track_color_sparse_custom(event, track_colors, overlap_color='black'):
    """
    Returns
    -------
    indices : (N_hits,)
    charges : (N_hits,) int labels (0 = overlap, 1..Ntracks = per-track)
    times   : (N_hits,) (dummy, here zeros)
    cmap    : ListedColormap with [overlap, track0, track1, ...]
    norm    : BoundaryNorm matching the integer labels
    vmin    : int, minimum label (0)
    vmax    : int, maximum label (n_colors-1)
    """

    # --- Step 1: extract track hits ---
    track_indices, overlap = prepare_event_tracks(event)
    n_tracks = len(track_indices)

    # If user gives too few colors, repeat
    if len(track_colors) < n_tracks:
        repeats = (n_tracks // len(track_colors)) + 1
        track_colors = (track_colors * repeats)[:n_tracks]

    # --- Step 2: convert colors → RGBA ---
    track_rgba = [mcolors.to_rgba(c) for c in track_colors]
    overlap_rgba = mcolors.to_rgba(overlap_color)

    # Colors: [overlap, track0, track1, ...]
    colors = [overlap_rgba] + track_rgba
    custom_cmap = ListedColormap(colors)

    n_colors = len(colors)

    # Boundaries between integer labels: -0.5, 0.5, 1.5, ..., (n_colors-0.5)
    boundaries = np.arange(-0.5, n_colors + 0.5, 1)
    norm = BoundaryNorm(boundaries, ncolors=n_colors)

    # --- Step 3: construct sparse arrays with integer labels ---
    sparse_indices = []
    sparse_charges = []   # these are actually category labels now
    sparse_times = []

    # Category 0 = overlap, 1..n_tracks = tracks
    overlap_label = 0

    for tid, idx in enumerate(track_indices):
        label = tid + 1  # track 0 → 1, track 1 → 2, ...
        for pmt in idx:
            if pmt in overlap:
                sparse_indices.append(pmt)
                sparse_charges.append(overlap_label)
                sparse_times.append(0)
            else:
                sparse_indices.append(pmt)
                sparse_charges.append(label)
                sparse_times.append(0)

    return (
        np.array(sparse_indices, dtype=int),
        np.array(sparse_charges, dtype=float),
        np.array(sparse_times, dtype=float),
        custom_cmap,
        0,              # vmin (min label)
        n_colors - 1    # vmax (max label)
    )


In [ ]:
from matplotlib.patches import Patch

indices, charges, times, cmap, vmin, vmax = \
    build_track_color_sparse_custom(
        all_events[0],
        track_colors=['cyan', 'blue'],
        overlap_color='red'
    )

legend_handles = [
    Patch(facecolor='cyan', label='Track 0'),
    Patch(facecolor='blue', label='Track 1'),
    Patch(facecolor='red', label='Overlap'),
]

detector_display(
    indices, charges + 1e-3, times,
    plot_time=False,
    colormap=cmap,
    show_colorbar=False,
    external_legend=legend_handles
)
